# Google Colab Host for EduMind AI Backend & Ollama

Run this single cell to clone the repo, mount Google Drive to pull source documents, automatically rebuild the vector store using GPU, install/start local Ollama, pull the LLM model, configure `.env` to point to localhost Ollama, and host the FastAPI backend server publicly using Ngrok Tunnel.

In [ ]:
# =====================================================================
# 1. Clone the GitHub Repository & Switch to Feature Branch
# =====================================================================
import os
import getpass
import shutil
import zipfile
import subprocess
import time
import re

repo_url = "github.com/jaynishthakar/demo-.git"
project_dir = "demo-"
branch_name = "feature/colab-tunnel-setup"

if not os.path.exists(project_dir):
    print("Cloning repository...")
    is_private = input("Is the GitHub repository private? (yes/no): ").strip().lower() == "yes"
    if is_private:
        token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ").strip()
        clone_url = f"https://{token}@{repo_url}"
    else:
        clone_url = f"https://{repo_url}"

    !git clone -b {branch_name} {clone_url}
else:
    print("Repository already exists. Pulling latest updates...")
    %cd {project_dir}
    !git checkout {branch_name}
    !git pull
    %cd ..

%cd {project_dir}

# =====================================================================
# 2. Install Python & System Dependencies
# =====================================================================
print("\nInstalling system dependencies (zstd)...")
!apt-get update && apt-get install -y zstd

print("\nInstalling python packages (requirements.txt)...")
!pip install -r requirements.txt

# =====================================================================
# 3. Mount Google Drive, Restore Docs, & Rebuild Vector Store
# =====================================================================
try:
    print("\nAttempting to mount Google Drive to check for source documents...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    zip_path = "/content/drive/MyDrive/VIT Final SOPs-01-07-2012.zip"
    if os.path.exists(zip_path):
        print("Extracting source documents directly from Google Drive...")
        staging_dir = "/content/demo-/data/staging"
        os.makedirs(staging_dir, exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            temp_extract = "/content/demo-/temp_docs"
            zip_ref.extractall(temp_extract)
            
            sub_dir = os.path.join(temp_extract, "VIT Final SOPs-01-07-2012")
            src_folder = sub_dir if os.path.exists(sub_dir) else temp_extract
            for file_name in os.listdir(src_folder):
                src_file = os.path.join(src_folder, file_name)
                dest_file = os.path.join(staging_dir, file_name)
                if os.path.isfile(src_file):
                    shutil.copy2(src_file, dest_file)
            
            shutil.rmtree(temp_extract)
            
        print("SUCCESS: Source documents staged in data/staging/.")
        
        print("\nBuilding SQLite database and generating GPU vector embeddings in ChromaDB...")
        subprocess.run(["python", "ingestion_pipeline.py"], check=True)
        subprocess.run(["python", "vector_store/index_pipeline.py"], check=True)
        print("SUCCESS: Vector store and metadata database are fully rebuilt!")
    else:
        print(f"\n[NOTE] 'VIT Final SOPs-01-07-2012.zip' not found in your Drive at: {zip_path}")
        print("Please place the ZIP file in your main Google Drive folder to automatically restore source documents.")
except Exception as e:
    print(f"\n[INFO] Google Drive mounting or auto-ingestion skipped/failed: {e}")
    print("Continuing backend startup without pre-ingested source documents.")

# =====================================================================
# 4. Install & Start Ollama (CORS Enabled)
# =====================================================================
print("\nInstalling Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\nStarting Ollama service in background...")
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("\nPulling Qwen2.5:7b model (takes ~1-2 mins)...")
!ollama pull qwen2.5:7b

# =====================================================================
# 5. Start the FastAPI Backend
# =====================================================================
print("\nStarting FastAPI Backend in the background...")
backend_env = os.environ.copy()
backend_env["OLLAMA_BASE_URL"] = "http://localhost:11434"
backend_env["LLM_BACKEND"] = "ollama"
backend_env["OLLAMA_MODEL"] = "qwen2.5:7b"

backend_log = open("backend_server.log", "w")
subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"],
    env=backend_env, stdout=backend_log, stderr=backend_log
)
time.sleep(8)

# =====================================================================
# 6. Install & Start Ngrok Tunnel on Backend Port (8000)
# =====================================================================
print("\nInstalling pyngrok...")
!pip install -q pyngrok

from pyngrok import ngrok
authtoken = "3FzpiDTu3BDjqq3oaOwhHZ210R8_2Gek4AzFydoBZwZA3GT6Z"
ngrok.set_auth_token(authtoken)

print("Starting Ngrok Tunnel...")
try:
    ngrok.kill() # Disconnect any old connections
    tunnel = ngrok.connect(8000, "http")
    print("\n" + "="*70)
    print(" SUCCESS: BACKEND IS NOW ONLINE VIA NGROK")
    print("-"*70)
    print(" Copy this URL into your Vercel Frontend UI:")
    print(f" {tunnel.public_url}")
    print("="*70 + "\n")
    
    # Keep cell alive
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping tunnel, backend, and Ollama...")
finally:
    try:
        ngrok.disconnect(tunnel.public_url)
    except:
        pass
    backend_process.terminate()
    ollama_proc.terminate()
    print("Cleanup complete.")
